# Gemma Model Benchmarking

This notebook benchmarks different sizes of the Gemma model by generating channel descriptions and measuring their alignment with the structural ground truth derived from video-title embeddings.

## 1) Setup & Dependencies

Install requirements, mount Drive, and set up Google Colab environment variables.

In [ ]:
!pip install -q pandas numpy scipy google-generativeai sentence-transformers scikit-learn

import os
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, spearmanr, kendalltau
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import google.generativeai as genai

try:
    from google.colab import userdata
    from google.colab import drive
    drive.mount('/content/drive')
    is_colab = True
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
except ImportError:
    print('Not running in Colab. Falling back to local execution.')
    is_colab = False

## 2) Load 20D Embeddings Data

Load the 20D video embeddings from the Graphiko exports. Provide a fallback for local testing.

In [ ]:
DATA_PATH = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv')

if not DATA_PATH.exists():
    print(f"Warning: {DATA_PATH} not found. Creating a dummy dataset.")
    dummy_data = []
    for i in range(100):
        dummy_data.append({
            'video_id': f'vid_{i}',
            'channel_name': f'Channel_{i % 10}',
            'video_title': f'Video Title {i}',
            **{f'embedding_reduced_{j:02d}': np.random.randn() for j in range(1, 21)}
        })
    df = pd.DataFrame(dummy_data)
else:
    df = pd.read_csv(DATA_PATH)

embedding_cols = [f'embedding_reduced_{i:02d}' for i in range(1, 21)]
df['embedding_20d'] = df[embedding_cols].values.tolist()

print(f"Loaded {len(df)} videos across {df['channel_name'].nunique()} channels.")

## 3) Compute Ground Truth Similarity Matrix

Compute the channel-by-channel cosine similarity matrix using the mean 20D video-title centroids.

In [ ]:
channel_centroids = df.groupby('channel_name')['embedding_20d'].apply(lambda x: np.mean(x.tolist(), axis=0))
channels = channel_centroids.index.tolist()
centroid_matrix = np.array(channel_centroids.tolist())

ground_truth_sim = cosine_similarity(centroid_matrix)

print(f"Ground truth matrix computed for {len(channels)} channels.")

## 4) Benchmarking Execution Loop

Generate descriptions using different Gemma models and calculate alignment with ground truth, following the metrics from the second notebook (Pearson, Spearman, Kendall, and Top-k overlap).

In [ ]:
MODELS = [
    'google/gemma-3-1b',
    'google/gemma-3-4b',
    'google/gemma-3-12b',
    'google/gemma-3-27b'
]

st_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_description(model_name, channel_name, titles):
    prompt = f"""
    You are an expert content analyst. Analyze the following list of video titles for the YouTube channel '{channel_name}'.
    Titles: {', '.join(titles[:50])}

    Provide a comprehensive summary of the channel's main themes and style.
    """
    if not is_colab:
        return f"Dummy description for {channel_name} using {model_name}."
    
    try:
        model = genai.GenerativeModel(model_name)
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        print(f"Error with model {model_name} for channel {channel_name}: {e}")
        return None

def get_top_k_neighbors(matrix, index, k):
    scores = matrix[index]
    # Exclude self by setting to -inf
    scores = scores.copy()
    scores[index] = -np.inf
    return np.argsort(scores)[::-1][:k]

results = []

for model_name in MODELS:
    print(f"Benchmarking {model_name}...")
    descriptions = []
    valid_channels = []
    
    for channel_name in channels:
        titles = df[df['channel_name'] == channel_name]['video_title'].tolist()
        desc = get_description(model_name, channel_name, titles)
        if desc:
            descriptions.append(desc)
            valid_channels.append(channel_name)
        time.sleep(1) # Rate limit protection
        
    if not descriptions:
        continue
        
    # Encode descriptions
    desc_embeddings = st_model.encode(descriptions)
    desc_sim = cosine_similarity(desc_embeddings)
    
    # Align with ground truth
    indices = [channels.index(c) for c in valid_channels]
    gt_subset = ground_truth_sim[np.ix_(indices, indices)]
    
    # Flatten upper triangles for correlation
    tri_idx = np.triu_indices(len(valid_channels), k=1)
    gt_flat = gt_subset[tri_idx]
    desc_flat = desc_sim[tri_idx]
    
    p_corr, _ = pearsonr(gt_flat, desc_flat)
    s_corr, _ = spearmanr(gt_flat, desc_flat)
    k_corr, _ = kendalltau(gt_flat, desc_flat)
    
    # Top-k overlap (e.g., k=5)
    k = 5
    overlaps = []
    for i in range(len(valid_channels)):
        gt_neighbors = set(get_top_k_neighbors(gt_subset, i, k))
        desc_neighbors = set(get_top_k_neighbors(desc_sim, i, k))
        overlaps.append(len(gt_neighbors & desc_neighbors) / k)
    mean_overlap = np.mean(overlaps)
    
    results.append({
        'model': model_name,
        'pearson': p_corr,
        'spearman': s_corr,
        'kendall': k_corr,
        'top_5_overlap': mean_overlap
    })

benchmark_df = pd.DataFrame(results)
print("Benchmark Results:")
print(benchmark_df)

## 5) Summary Table

Final results comparison.

In [ ]:
import matplotlib.pyplot as plt

if not benchmark_df.empty:
    benchmark_df.plot(kind='bar', x='model', y=['pearson', 'spearman', 'kendall', 'top_5_overlap'], figsize=(12, 6))
    plt.title('Gemma Model Performance Alignment (All Metrics)')
    plt.ylabel('Score')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()